In [4]:
# ============================================================
# Get Fit with Drishti
# Phase 6 Part A: Simple Tableau Export Files
# ============================================================

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Set folder paths
# ------------------------------------------------------------
project_root = Path(".")

clean_data_path = project_root / "02_cleaned_data" / "Drishti_Clean_Data.csv"

output_folder = project_root / "07_tableau_dashboard" / "tableau_exports"
output_folder.mkdir(parents=True, exist_ok=True)

print("Clean data path:")
print(clean_data_path)

print("\nOutput folder:")
print(output_folder)


# ------------------------------------------------------------
# 2. Load cleaned data
# ------------------------------------------------------------

df = pd.read_csv(clean_data_path)

print("\nClean data loaded successfully.")
print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 3. Convert date columns
# ------------------------------------------------------------

df["Payment_Date"] = pd.to_datetime(df["Payment_Date"])
df["Program_Start_Date"] = pd.to_datetime(df["Program_Start_Date"])
df["Program_End_Date"] = pd.to_datetime(df["Program_End_Date"])


# ------------------------------------------------------------
# 4. Create extra columns for Tableau
# ------------------------------------------------------------

goal_map = {
    "No": 0,
    "Partial": 1,
    "Ongoing": 2,
    "Yes": 3
}

df["Goal_Achieved_Encoded"] = df["Goal_Achieved"].map(goal_map)

df["Churned"] = df["Status"].isin(["Ghost", "Inactive", "Quit"])

df["Superfan_Score"] = (
    (df["Goal_Achieved"] == "Yes").astype(int) * 30 +
    (df["Second_Round"] == "Yes").astype(int) * 25 +
    (df["Progress_Photos_Shared"] == "Yes").astype(int) * 20 +
    (df["Checkin_Rate_Pct"] >= 80).astype(int) * 15 +
    (df["Lead_Source"] == "Referral").astype(int) * 10
)

df["Churn_Probability"] = (
    (df["Days_To_First_Engagement"].astype(str) == "Never").astype(int) * 0.45 +
    (df["Checkin_Rate_Pct"] < 50).astype(int) * 0.25 +
    (df["Week3_Still_Active"] == "No").astype(int) * 0.30
)

df["Churn_Probability"] = df["Churn_Probability"].round(2)

if "Client_Persona" not in df.columns:
    df["Client_Persona"] = "Not Added Yet"


# ------------------------------------------------------------
# 5. Export tableau_clients.csv
# ------------------------------------------------------------

df.to_csv(output_folder / "tableau_clients.csv", index=False)


# ------------------------------------------------------------
# 6. Export tableau_monthly.csv
# ------------------------------------------------------------

df["Payment_Month"] = df["Payment_Date"].dt.to_period("M").dt.to_timestamp()

tableau_monthly = (
    df.groupby(["Payment_Month", "Lead_Source"], as_index=False)
    .agg(
        Client_Count=("Client_ID", "count"),
        Revenue_USD=("Revenue_USD", "sum")
    )
)

tableau_monthly.to_csv(output_folder / "tableau_monthly.csv", index=False)


# ------------------------------------------------------------
# 7. Export tableau_funnel.csv
# ------------------------------------------------------------

tableau_funnel = pd.DataFrame({
    "Stage_Number": [1, 2, 3, 4, 5, 6],
    "Funnel_Stage": [
        "Paid Clients",
        "First Engagement",
        "Active at Week 3",
        "Completed Program",
        "Achieved or Partially Achieved Goal",
        "Second Round"
    ],
    "Client_Count": [
        df["Client_ID"].nunique(),
        df[df["Days_To_First_Engagement"].astype(str) != "Never"]["Client_ID"].nunique(),
        df[df["Week3_Still_Active"] == "Yes"]["Client_ID"].nunique(),
        df[df["Status"] == "Completed"]["Client_ID"].nunique(),
        df[df["Goal_Achieved"].isin(["Yes", "Partial"])]["Client_ID"].nunique(),
        df[df["Second_Round"] == "Yes"]["Client_ID"].nunique()
    ]
})

tableau_funnel["Dropoff_From_Previous_Stage"] = (
    tableau_funnel["Client_Count"].shift(1) - tableau_funnel["Client_Count"]
).fillna(0)

tableau_funnel["Stage_Conversion_Rate_Pct"] = (
    tableau_funnel["Client_Count"] / tableau_funnel["Client_Count"].shift(1) * 100
).fillna(100).round(2)

tableau_funnel.to_csv(output_folder / "tableau_funnel.csv", index=False)


# ------------------------------------------------------------
# 8. Final check
# ------------------------------------------------------------

print("\nPhase 6 Part A completed successfully.")
print("Files saved in:", output_folder)

display(tableau_funnel)

Clean data path:
02_cleaned_data/Drishti_Clean_Data.csv

Output folder:
07_tableau_dashboard/tableau_exports

Clean data loaded successfully.
Dataset shape: (3000, 40)

Phase 6 Part A completed successfully.
Files saved in: 07_tableau_dashboard/tableau_exports


,Stage_Number,Funnel_Stage,Client_Count,Dropoff_From_Previous_Stage,Stage_Conversion_Rate_Pct
0,1,Paid Clients,3000,0.0,100.00
1,2,First Engagement,3000,0.0,100.00
2,3,Active at Week 3,2677,323.0,89.23
3,4,Completed Program,2431,246.0,90.81
4,5,Achieved or Partially Achieved Goal,2431,0.0,100.00
5,6,Second Round,428,2003.0,17.61
